In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# Replace with your source JSON path
json_path = Path("/Users/qmou/App/yelp_academic_dataset_review.json")
parquet_path = json_path.with_suffix(".parquet")

chunk_size = 100_000
writer = None
total_rows = 0

try:
    for chunk in pd.read_json(json_path, lines=True, chunksize=chunk_size):
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(parquet_path, table.schema, compression="snappy")
        writer.write_table(table)
        total_rows += len(chunk)
        print(f"Wrote {total_rows:,} rows...")
finally:
    if writer is not None:
        writer.close()

print(f"Done. Output: {parquet_path} ({total_rows:,} rows)")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from pathlib import Path

# Replace with your source parquet path (the output of the cell above)
src = Path("/Users/qmou/App/yelp_academic_dataset_review.parquet")
out_dir = src.parent

shards = [
    ("2005-2014", 2005, 2014),
    ("2015-2016", 2015, 2016),
    ("2017-2018", 2017, 2018),
    ("2019",      2019, 2019),
    ("2020-2022", 2020, 2022),
]

pf = pq.ParquetFile(src)
writers = {}
counts = {name: 0 for name, _, _ in shards}

try:
    for batch in pf.iter_batches(batch_size=100_000):
        table = pa.Table.from_batches([batch])
        years = pc.year(pc.cast(table["date"], "timestamp[s]"))
        for name, lo, hi in shards:
            mask = pc.and_(pc.greater_equal(years, lo), pc.less_equal(years, hi))
            filtered = table.filter(mask)
            if len(filtered) == 0:
                continue
            if name not in writers:
                path = out_dir / f"yelp_reviews_{name}.parquet"
                writers[name] = pq.ParquetWriter(path, filtered.schema, compression="zstd")
            writers[name].write_table(filtered)
            counts[name] += len(filtered)
finally:
    for w in writers.values():
        w.close()

for name, n in counts.items():
    path = out_dir / f"yelp_reviews_{name}.parquet"
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"yelp_reviews_{name}.parquet: {n:,} rows, {size_mb:.1f} MB")